# HW2 — Инженерная реализация ускорения модели

**Модель из ДЗ-1:** `BERT-base-uncased` для классификации текстовых обращений.

**Метод ускорения:** `knowledge distillation` (teacher-student), где student = `DistilBERT`.

В ноутбуке реализованы:
1. baseline-конфигурация и bottlenecks;
2. корректная реализация дистилляции;
3. воспроизводимый эксперимент;
4. сравнение качества и производительности;
5. инженерный вывод о применимости в production.

## 1. Базовая модель, сценарий и bottlenecks

- **Задача:** многоклассовая классификация текстов (датасет `AG News`, 4 класса).
- **Сценарий:** inference в near-real-time.
- **Целевая платформа:** GPU в Colab (обычно NVIDIA T4), что соответствует серверному inference-профилю.
- **Baseline:** `bert-base-uncased` + классификационная голова.
- **Ключевые bottlenecks (из ДЗ-1):**
  - высокая вычислительная стоимость трансформерных блоков;
  - большой объем параметров (память модели);
  - латентность инференса при небольшом batch size.

Выбранная гипотеза ускорения: уменьшить глубину и размер модели через дистилляцию, сохраняя качество на прикладном уровне.

In [1]:
# Если запускаете в Colab: раскомментируйте установку зависимостей.
!pip -q uninstall -y transformers tokenizers
!pip -q install -U "transformers>=4.48,<4.53" "tokenizers>=0.21,<0.22" datasets accelerate evaluate scikit-learn pandas

import os
import time
import math
import random
import tempfile
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 25.10.

In [2]:
@dataclass
class CFG:
    teacher_name: str = 'bert-base-uncased'
    student_name: str = 'distilbert-base-uncased'
    num_labels: int = 4
    max_length: int = 128
    train_subset: int = 12000      # можно увеличить для лучшего качества
    test_subset: int = 3000
    batch_size_train: int = 16
    batch_size_eval: int = 32
    teacher_epochs: int = 1
    student_epochs: int = 1
    lr_teacher: float = 2e-5
    lr_student: float = 3e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    temperature: float = 2.0
    alpha_ce: float = 0.5          # вес supervised CE
    alpha_kd: float = 0.5          # вес distillation KD loss
    save_dir: str = './checkpoints_hw2'

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)
cfg


CFG(teacher_name='bert-base-uncased', student_name='distilbert-base-uncased', num_labels=4, max_length=128, train_subset=12000, test_subset=3000, batch_size_train=16, batch_size_eval=32, teacher_epochs=1, student_epochs=1, lr_teacher=2e-05, lr_student=3e-05, weight_decay=0.01, warmup_ratio=0.06, temperature=2.0, alpha_ce=0.5, alpha_kd=0.5, save_dir='./checkpoints_hw2')

## 2. Подготовка данных и единый воспроизводимый пайплайн

Для корректного сравнения baseline и ускоренной модели используются одинаковые:
- датасет;
- train/test разбиение;
- токенизация и `max_length`;
- метрики и процедура бенчмарка.


In [3]:
dataset = load_dataset('ag_news')
dataset

tokenizer = AutoTokenizer.from_pretrained(cfg.teacher_name)

def preprocess(batch):
    return tokenizer(batch['text'], truncation=True, max_length=cfg.max_length)

tokenized = dataset.map(preprocess, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_ds = tokenized['train'].shuffle(seed=42).select(range(cfg.train_subset))
test_ds = tokenized['test'].shuffle(seed=42).select(range(cfg.test_subset))

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size_train, shuffle=True, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size_eval, shuffle=False, collate_fn=collator)

print(f'Train subset: {len(train_ds)} | Test subset: {len(test_ds)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Train subset: 12000 | Test subset: 3000


In [4]:
def count_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model: nn.Module):
    total_bytes = sum(p.nelement() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        total_loss += out.loss.item()
        n_batches += 1

        preds = torch.argmax(out.logits, dim=-1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch['labels'].detach().cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1w = f1_score(all_labels, all_preds, average='weighted')
    return {
        'loss': total_loss / max(n_batches, 1),
        'accuracy': acc,
        'f1_weighted': f1w,
    }

def train_supervised(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, lr: float, epochs: int):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    total_steps = epochs * len(train_loader)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            if step % 200 == 0:
                print(f'[Teacher][Epoch {epoch}] step {step}/{len(train_loader)} loss={running_loss/step:.4f}')

        val = evaluate_model(model, test_loader)
        print(f"[Teacher][Epoch {epoch}] val_loss={val['loss']:.4f} acc={val['accuracy']:.4f} f1w={val['f1_weighted']:.4f}")

    return model

def distillation_loss(student_logits, teacher_logits, true_labels, T=2.0, alpha_ce=0.5, alpha_kd=0.5):
    ce = nn.CrossEntropyLoss()(student_logits, true_labels)
    kd = nn.KLDivLoss(reduction='batchmean')(
        nn.functional.log_softmax(student_logits / T, dim=-1),
        nn.functional.softmax(teacher_logits / T, dim=-1)
    ) * (T * T)
    return alpha_ce * ce + alpha_kd * kd, ce.detach(), kd.detach()

def train_student_with_distillation(student: nn.Module, teacher: nn.Module, train_loader: DataLoader, test_loader: DataLoader):
    student.to(device)
    teacher.to(device)
    teacher.eval()

    optimizer = AdamW(student.parameters(), lr=cfg.lr_student, weight_decay=cfg.weight_decay)
    total_steps = cfg.student_epochs * len(train_loader)
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    for epoch in range(1, cfg.student_epochs + 1):
        student.train()
        loss_sum, ce_sum, kd_sum = 0.0, 0.0, 0.0
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                teacher_logits = teacher(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits

            student_logits = student(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
            loss, ce_part, kd_part = distillation_loss(
                student_logits,
                teacher_logits,
                batch['labels'],
                T=cfg.temperature,
                alpha_ce=cfg.alpha_ce,
                alpha_kd=cfg.alpha_kd,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            loss_sum += loss.item()
            ce_sum += ce_part.item()
            kd_sum += kd_part.item()

            if step % 200 == 0:
                print(
                    f"[Student KD][Epoch {epoch}] step {step}/{len(train_loader)} "
                    f"loss={loss_sum/step:.4f} ce={ce_sum/step:.4f} kd={kd_sum/step:.4f}"
                )

        val = evaluate_model(student, test_loader)
        print(f"[Student KD][Epoch {epoch}] val_loss={val['loss']:.4f} acc={val['accuracy']:.4f} f1w={val['f1_weighted']:.4f}")

    return student


## 3. Реализация ускорения: Knowledge Distillation

Ниже реализуется классическая схема teacher-student:
- **Teacher:** `BERT-base-uncased` (более точная, но тяжелая модель).
- **Student:** `DistilBERT` (меньше слоев и параметров).
- **Objective:** комбинация `CrossEntropy` (по истинным меткам) и `KL-divergence` (по soft targets teacher).

Таким образом ускорение достигается через:
1. уменьшение числа параметров;
2. снижение объема вычислений (FLOPs);
3. уменьшение латентности и размера модели.

In [5]:
# --- Teacher baseline training ---
teacher = AutoModelForSequenceClassification.from_pretrained(cfg.teacher_name, num_labels=cfg.num_labels)
teacher = train_supervised(
    model=teacher,
    train_loader=train_loader,
    test_loader=test_loader,
    lr=cfg.lr_teacher,
    epochs=cfg.teacher_epochs,
)

teacher_metrics = evaluate_model(teacher, test_loader)
print('Teacher final metrics:', teacher_metrics)

teacher_ckpt = os.path.join(cfg.save_dir, 'teacher_bert_agnews.pt')
torch.save(teacher.state_dict(), teacher_ckpt)
print('Saved teacher checkpoint:', teacher_ckpt)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Teacher][Epoch 1] step 200/750 loss=0.6155
[Teacher][Epoch 1] step 400/750 loss=0.4655
[Teacher][Epoch 1] step 600/750 loss=0.4069
[Teacher][Epoch 1] val_loss=0.2451 acc=0.9150 f1w=0.9148
Teacher final metrics: {'loss': 0.24510030602996655, 'accuracy': 0.915, 'f1_weighted': 0.9147606950819602}
Saved teacher checkpoint: ./checkpoints_hw2/teacher_bert_agnews.pt


In [6]:
# --- Student training with distillation ---
student = AutoModelForSequenceClassification.from_pretrained(cfg.student_name, num_labels=cfg.num_labels)
student = train_student_with_distillation(student, teacher, train_loader, test_loader)

student_metrics = evaluate_model(student, test_loader)
print('Student final metrics:', student_metrics)

student_ckpt = os.path.join(cfg.save_dir, 'student_distilbert_agnews_kd.pt')
torch.save(student.state_dict(), student_ckpt)
print('Saved student checkpoint:', student_ckpt)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Student KD][Epoch 1] step 200/750 loss=0.6346 ce=0.5690 kd=0.7003
[Student KD][Epoch 1] step 400/750 loss=0.4170 ce=0.4276 kd=0.4063
[Student KD][Epoch 1] step 600/750 loss=0.3397 ce=0.3795 kd=0.2999
[Student KD][Epoch 1] val_loss=0.2539 acc=0.9153 f1w=0.9150
Student final metrics: {'loss': 0.25394774705884937, 'accuracy': 0.9153333333333333, 'f1_weighted': 0.9150256653534452}
Saved student checkpoint: ./checkpoints_hw2/student_distilbert_agnews_kd.pt


## 4. Метрики производительности и бенчмарк inference

Используются метрики, релевантные production inference:
- **Качество:** `Accuracy`, `Weighted F1`.
- **Скорость:** средняя latency (мс) и throughput (samples/sec).
- **Ресурсы:** число параметров, размер модели в памяти, пиковая GPU-память в инференсе.

Замеры baseline и student проводятся в одинаковых условиях (одно и то же устройство, одинаковый batch и sequence length).

In [7]:
@torch.no_grad()
def benchmark_inference(model: nn.Module, loader: DataLoader, n_warmup: int = 20, n_iters: int = 100):
    model.eval().to(device)
    batch = next(iter(loader))
    batch = {k: v.to(device) for k, v in batch.items()}

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # Warmup
    for _ in range(n_warmup):
        _ = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    for _ in range(n_iters):
        if torch.cuda.is_available():
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            _ = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            end.record()
            torch.cuda.synchronize()
            elapsed_ms = start.elapsed_time(end)
        else:
            t0 = time.perf_counter()
            _ = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
        times.append(elapsed_ms)

    latency_mean = float(np.mean(times))
    latency_p50 = float(np.percentile(times, 50))
    latency_p95 = float(np.percentile(times, 95))
    batch_size = batch['input_ids'].shape[0]
    throughput = batch_size / (latency_mean / 1000.0)

    peak_gpu_mem_mb = 0.0
    if torch.cuda.is_available():
        peak_gpu_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return {
        'latency_ms_mean': latency_mean,
        'latency_ms_p50': latency_p50,
        'latency_ms_p95': latency_p95,
        'throughput_samples_per_sec': throughput,
        'peak_gpu_mem_mb': peak_gpu_mem_mb,
        'bench_batch_size': batch_size,
    }

teacher_perf = benchmark_inference(teacher, test_loader)
student_perf = benchmark_inference(student, test_loader)

teacher_stats = {
    'model': 'Teacher BERT-base',
    'params_million': count_params(teacher) / 1e6,
    'model_size_mb': model_size_mb(teacher),
    'accuracy': teacher_metrics['accuracy'],
    'f1_weighted': teacher_metrics['f1_weighted'],
    **teacher_perf,
}

student_stats = {
    'model': 'Student DistilBERT (KD)',
    'params_million': count_params(student) / 1e6,
    'model_size_mb': model_size_mb(student),
    'accuracy': student_metrics['accuracy'],
    'f1_weighted': student_metrics['f1_weighted'],
    **student_perf,
}

results_df = pd.DataFrame([teacher_stats, student_stats])
results_df


,model,params_million,model_size_mb,accuracy,f1_weighted,latency_ms_mean,latency_ms_p50,latency_ms_p95,throughput_samples_per_sec,peak_gpu_mem_mb,bench_batch_size
0,Teacher BERT-base,109.485316,417.653336,0.915000,0.914761,155.573150,155.110001,160.692545,205.691020,1469.014648,32
1,Student DistilBERT (KD),66.956548,255.418961,0.915333,0.915026,80.858609,80.948769,81.563755,395.752541,1469.342773,32


In [8]:
# Производные инженерные метрики (выигрыш/компромиссы)
t = results_df.iloc[0]
s = results_df.iloc[1]

summary = {
    'speedup_latency_mean_x': t['latency_ms_mean'] / s['latency_ms_mean'],
    'throughput_gain_x': s['throughput_samples_per_sec'] / t['throughput_samples_per_sec'],
    'param_reduction_x': t['params_million'] / s['params_million'],
    'model_size_reduction_x': t['model_size_mb'] / s['model_size_mb'],
    'accuracy_delta_abs': s['accuracy'] - t['accuracy'],
    'f1_weighted_delta_abs': s['f1_weighted'] - t['f1_weighted'],
}

summary_df = pd.DataFrame([summary])
summary_df


,speedup_latency_mean_x,throughput_gain_x,param_reduction_x,model_size_reduction_x,accuracy_delta_abs,f1_weighted_delta_abs
0,1.924015,1.924015,1.63517,1.63517,0.000333,0.000265


## 5. Интерпретация результатов

По результатам эксперимента (Teacher BERT-base vs Student DistilBERT с knowledge distillation на AG News):

- **Параметры:** Student уменьшил количество параметров примерно в **1.64** раз (109.5M → 67.0M).
- **Размер модели:** Уменьшился в **1.64** раз (418 MB → 255 MB).
- **Латентность:** Средняя latency улучшилась в **~1.92** раз (155.6 мс → 80.9 мс при batch=32).
- **Пропускная способность:** Throughput вырос в **~1.92** раз (206 → 396 samples/sec).
- **Качество:** В данном запуске потери качества **нет**: Accuracy и Weighted F1 у student чуть выше (Δacc ≈ +0.03%, ΔF1 ≈ +0.03%), что укладывается в дисперсию при одной эпохе и фиксированном seed.

**Инженерная трактовка:**

1. Дистилляция дала почти двукратное ускорение и сокращение модели при сохранении (и даже лёгком росте) метрик — для inference-сценария это **целесообразно**.
2. Если в других запусках или на других данных появится заметная потеря F1, имеет смысл: увеличить обучающую выборку, число эпох student, подобрать `temperature` и веса `alpha_ce`/`alpha_kd`, при необходимости добавить аугментации.


## 6. Итоговый инженерный вывод

**Целесообразность метода:**  
Knowledge distillation для BERT в данном эксперименте **оправдана**: при сокращении модели в ~1.64 раза и ускорении инференса в ~1.92 раза качество (Accuracy и Weighted F1) не ухудшилось, а слегка выросло в пределах статистической погрешности.

**Что выиграли (по замерам):**
- **Латентность:** снижение с ~156 мс до ~81 мс на batch=32 (~1.92×).
- **Throughput:** рост с ~206 до ~396 samples/sec (~1.92×).
- **Размер модели:** с ~418 MB до ~255 MB (~1.64×), меньше параметров (109.5M → 67.0M).

**Чем заплатили:**
- Дополнительная стадия обучения student с дистилляцией (и необходимость обучить/загрузить teacher).
- Потенциальная потеря качества при других данных или настройках (здесь не проявилась).
- Усложнение пайплайна: два типа моделей, подбор temperature и весов CE/KD.

**Когда применять в production:**  
При ограничениях по latency и бюджету GPU/CPU и при допустимой небольшой (или нулевой) деградации качества — как в нашем случае.

**Когда может быть неэффективно:**  
Когда критична максимальная точность без потерь или нет ресурсов на обучение и сопровождение distillation-пайплайна (teacher + student, гиперпараметры).